In [2]:
import sqlite3
import pandas as pd
import os
from pathlib import Path

In [3]:
# Base paths
BASE_DIR = Path("/home/agrupa-lab/agrupa")
DB_PATH = BASE_DIR / "agrupa.sqlite"
DATA_RAW = BASE_DIR / "data_raw"

print("Database exists:", DB_PATH.exists())
print("Data raw exists:", DATA_RAW.exists())

Database exists: True
Data raw exists: True


In [4]:
conn = sqlite3.connect(DB_PATH)

# Get all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

Tables in database:
              name
0          artwork
1         icon_tag
2  sqlite_sequence
3      source_file
4  artwork_tag_raw
5      artwork_tag
6      tag_closure
7    artwork_image
8  qwen_captions_m
9        figures_m


In [5]:
for table in tables['name']:
    print(f"\n{'='*50}")
    print(f"TABLE: {table}")
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn)
    print(f"Shape: {df.shape}")
    print(df.head())


TABLE: artwork
Shape: (5, 25)
    cat_no                                             titulo  \
0  P000002                                 El juicio de Paris   
1  P000006   Sagrada Familia y el cardenal Fernando de Medici   
2  P000013           Rendición de Sevilla al rey san Fernando   
3  P000014  María Isabel de Borbón y Sajonia, infanta de N...   
4  P000015                                     La Anunciación   

                     autor escuela_obra  \
0        Albani, Francesco     Italiana   
1       Allori, Alessandro     Italiana   
2  Flipart, Charles-Joseph     Francesa   
3           Ruta, Clemente     Italiana   
4            Angelico, Fra      Toscana   

                                           ubicacion  \
0  Sala 6 (Planta 1, Sector Norte) - Edificio Vil...   
1                                          (Almacén)   
2                                          (Almacén)   
3                                          (Almacén)   
4  Sala 56 B (Planta 0, Sector Centro-N

In [6]:
artworks = pd.read_sql("SELECT * FROM artwork", conn)
print(f"Total artworks in DB: {len(artworks)}")
print(f"\nColumns: {list(artworks.columns)}")
print(artworks.head(10))

Total artworks in DB: 15455

Columns: ['cat_no', 'titulo', 'autor', 'escuela_obra', 'ubicacion', 'medidas', 'tecnicas', 'soporte', 'materia', 'otros_no_inv', 'fecha_ingreso', 'forma_ingreso', 'procedencia', 'tipo_objeto', 'datacion', 'tema', 'pais_ubicacion', 'comunidad_ubicacion', 'provincia_ubicacion', 'localidad_ubicacion', 'area_departamento', 'marco_obra_actual', 'is_religious', 'is_fauna', 'century']
        cat_no                                             titulo  \
0      P000002                                 El juicio de Paris   
1      P000006   Sagrada Familia y el cardenal Fernando de Medici   
2      P000013           Rendición de Sevilla al rey san Fernando   
3      P000014  María Isabel de Borbón y Sajonia, infanta de N...   
4      P000015                                     La Anunciación   
5  P000015/001                                     La Anunciación   
6      P000016                    Pietro Manna, médico de Cremona   
7      P000018                        

In [7]:
images = pd.read_sql("SELECT * FROM artwork_image", conn)
print(f"Total artworks with images: {len(images)}")

# Artworks missing images
missing = artworks[~artworks['cat_no'].isin(images['cat_no'])]
print(f"Artworks missing images: {len(missing)}")
print(f"Image coverage: {len(images)/len(artworks)*100:.1f}%")

Total artworks with images: 7429
Artworks missing images: 8026
Image coverage: 48.1%


In [8]:
# List all Excel files
excel_files = list(DATA_RAW.glob("*.xlsx"))
print(f"Total Excel files found: {len(excel_files)}")

# Categorize them
fauna = [f for f in excel_files if f.name.startswith("Fauna")]
personas = [f for f in excel_files if f.name.startswith("Personas")]
temas = [f for f in excel_files if f.name.startswith("Temas")]

print(f"  Fauna files: {len(fauna)}")
print(f"  Personas files: {len(personas)}")
print(f"  Temas files: {len(temas)}")

Total Excel files found: 362
  Fauna files: 214
  Personas files: 1
  Temas files: 147


In [9]:
# Load the Personas file as a sample
personas_df = pd.read_excel(DATA_RAW / "Personas.xlsx")
print(f"Personas shape: {personas_df.shape}")
print(f"Columns: {list(personas_df.columns)}")
print(personas_df.head())

Personas shape: (14922, 22)
Columns: ['N. Cat.', 'Título', 'Autor', 'Escuela Obra', 'Ubicación', 'Medidas', 'Técnicas', 'Soporte', 'Materia', 'Otros Nº Inv.', 'Fecha Ingreso', 'Forma Ingreso', 'Procedencia', 'Tipo Objeto', 'Datación', 'Tema', 'País Ubicación', 'Comunidad Ubicación', 'Provincia Ubicación', 'Localidad Ubicación', 'Área / Departamento', 'Marco/Obra Actual']
   N. Cat.                           Título  \
0      NaN                              NaN   
1  P000001              El tocador de Venus   
2  P000002               El juicio de Paris   
3  P000003  Nacimiento de san Juan Bautista   
4  P000004       El Nacimiento de la Virgen   

                                               Autor Escuela Obra  \
0                                                NaN          NaN   
1                                  Albani, Francesco     Italiana   
2                                  Albani, Francesco     Italiana   
3                                     Sacchi, Andrea     Italiana  

In [10]:
tags_raw = pd.read_sql("SELECT * FROM artwork_tag_raw LIMIT 20", conn)
print("Raw tags sample:")
print(tags_raw)

tags = pd.read_sql("SELECT * FROM artwork_tag LIMIT 20", conn)
print("\nProcessed tags sample:")
print(tags)

Raw tags sample:
         cat_no  tag_id  source_id
0       P000002       1          1
1       P000006       1          1
2       P000013       1          1
3       P000014       1          1
4       P000015       1          1
5   P000015/001       1          1
6       P000016       1          1
7       P000018       1          1
8       P000019       1          1
9       P000021       1          1
10      P000022       1          1
11      P000023       1          1
12      P000024       1          1
13      P000026       1          1
14      P000027       1          1
15      P000028       1          1
16      P000029       1          1
17      P000030       1          1
18      P000031       1          1
19      P000033       1          1

Processed tags sample:
     cat_no  tag_id
0   P000002     133
1   P000002     237
2   P000002     275
3   P000002     276
4   P000002     277
5   P000002     217
6   P000002     125
7   P000002     351
8   P000006     400
9   P000006      55
10  

In [11]:
print("=" * 50)
print("DATA AUDIT SUMMARY")
print("=" * 50)
print(f"Total artworks in DB:        {len(artworks)}")
print(f"Artworks with images:        {len(images)}")
print(f"Artworks missing images:     {len(missing)}")
print(f"Image coverage:              {len(images)/len(artworks)*100:.1f}%")
print(f"Total raw Excel files:       {len(excel_files)}")
print(f"  - Fauna:                   {len(fauna)}")
print(f"  - Personas:                {len(personas)}")
print(f"  - Temas:                   {len(temas)}")
print("=" * 50)

conn.close()

DATA AUDIT SUMMARY
Total artworks in DB:        15455
Artworks with images:        7429
Artworks missing images:     8026
Image coverage:              48.1%
Total raw Excel files:       362
  - Fauna:                   214
  - Personas:                1
  - Temas:                   147


In [12]:
conn = sqlite3.connect(DB_PATH)

# Check the cat_no formats in each table
artwork_ids = pd.read_sql("SELECT DISTINCT substr(cat_no,1,1) as prefix, COUNT(*) as count FROM artwork GROUP BY prefix", conn)
image_ids = pd.read_sql("SELECT DISTINCT substr(cat_no,1,1) as prefix, COUNT(*) as count FROM artwork_image GROUP BY prefix", conn)

print("Artwork ID prefixes:")
print(artwork_ids)
print("\nImage ID prefixes:")
print(image_ids)

# Check overlap
overlap = pd.read_sql("""
    SELECT COUNT(*) as matched 
    FROM artwork a 
    INNER JOIN artwork_image i ON a.cat_no = i.cat_no
""", conn)
print("\nDirect ID matches between artwork and image tables:")
print(overlap)

conn.close()

Artwork ID prefixes:
  prefix  count
0      G     36
1      H   8044
2      P   7375

Image ID prefixes:
  prefix  count
0      G     15
1      H   1148
2      P   6266

Direct ID matches between artwork and image tables:
   matched
0     7429


In [13]:
conn = sqlite3.connect(DB_PATH)

# Check coverage by prefix
coverage = pd.read_sql("""
    SELECT 
        substr(a.cat_no,1,1) as prefix,
        COUNT(a.cat_no) as total_artworks,
        COUNT(i.cat_no) as has_image,
        ROUND(COUNT(i.cat_no) * 100.0 / COUNT(a.cat_no), 1) as coverage_pct
    FROM artwork a
    LEFT JOIN artwork_image i ON a.cat_no = i.cat_no
    GROUP BY prefix
""", conn)
print(coverage)

# Also check: of the P artworks, how many are fauna vs human scenes?
fauna_humans = pd.read_sql("""
    SELECT 
        is_fauna,
        is_religious,
        COUNT(*) as count
    FROM artwork
    WHERE substr(cat_no,1,1) = 'P'
    GROUP BY is_fauna, is_religious
""", conn)
print("\nP artworks breakdown:")
print(fauna_humans)

conn.close()

  prefix  total_artworks  has_image  coverage_pct
0      G              36         15          41.7
1      H            8044       1148          14.3
2      P            7375       6266          85.0

P artworks breakdown:
   is_fauna  is_religious  count
0         0             0   2857
1         0             1   1976
2         1             0   1667
3         1             1    875


In [14]:
conn = sqlite3.connect(DB_PATH)

# Save the core dataset stats
summary = {
    "total_artworks": 15455,
    "P_paintings_total": 7375,
    "P_paintings_with_images": 6266,
    "P_image_coverage_pct": 85.0,
    "human_scenes": 4833,
    "fauna_scenes": 2542,
    "ready_for_mllm": 6266
}

import json
output_path = Path("/home/agrupa-lab/agrupa/IE_capstones/Omar/audit_summary.json")
with open(output_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Audit complete. Summary saved.")
print(json.dumps(summary, indent=4))

conn.close()

Audit complete. Summary saved.
{
    "total_artworks": 15455,
    "P_paintings_total": 7375,
    "P_paintings_with_images": 6266,
    "P_image_coverage_pct": 85.0,
    "human_scenes": 4833,
    "fauna_scenes": 2542,
    "ready_for_mllm": 6266
}


In [15]:
conn = sqlite3.connect(DB_PATH)

# See what tags exist and how they're structured
tags = pd.read_sql("""
    SELECT t.tag_id, t.name, t.slug, t.depth, t.parent_id
    FROM icon_tag t
    ORDER BY t.depth, t.parent_id, t.tag_id
    LIMIT 50
""", conn)

print(tags.to_string())
conn.close()

    tag_id                           name                                 slug  depth  parent_id
0        1                          Fauna                                fauna      0        NaN
1      237                       Personas                             personas      0        NaN
2      238                          Temas                                temas      0        NaN
3        2                       Anfibios                       fauna/anfibios      1        1.0
4        6                       Animales                       fauna/animales      1        1.0
5       23                     Artropodos                     fauna/artropodos      1        1.0
6       24                           Aves                           fauna/aves      1        1.0
7      165                      Mamiferos                      fauna/mamiferos      1        1.0
8      219                          Peces                          fauna/peces      1        1.0
9      230                    

In [16]:
conn = sqlite3.connect(DB_PATH)

# Get all profession and portrait related tags
human_tags = pd.read_sql("""
    SELECT t.tag_id, t.name, t.slug, t.depth, t.parent_id
    FROM icon_tag t
    WHERE t.slug LIKE 'temas/oficiosyprofesiones%'
    OR t.slug LIKE 'temas/retrato%'
    OR t.slug LIKE 'temas/militar%'
    ORDER BY t.slug
""", conn)

print(human_tags.to_string())
conn.close()

    tag_id                                            name                                                         slug  depth  parent_id
0      343                                         Militar                                                temas/militar      1        238
1      344                                           Armas                                          temas/militar/armas      2        343
2      345                                         Desfile                                        temas/militar/desfile      2        343
3      346                                Guerra-y-Combate                               temas/militar/guerra-y-combate      2        343
4      347                                       Insignias                                      temas/militar/insignias      2        343
5      356                             OficiosYProfesiones                                    temas/oficiosyprofesiones      1        238
6      357                        

In [17]:
conn = sqlite3.connect(DB_PATH)

mammal_tags = pd.read_sql("""
    SELECT t.tag_id, t.name, t.slug, t.depth, t.parent_id
    FROM icon_tag t
    WHERE t.slug LIKE 'fauna/mamiferos%'
    ORDER BY t.slug
""", conn)

print(mammal_tags.to_string())
conn.close()

    tag_id                                    name                                                  slug  depth  parent_id
0      165                               Mamiferos                                       fauna/mamiferos      1          1
1      166                                Antilope                              fauna/mamiferos/antilope      2        165
2      167                                 Ardilla                               fauna/mamiferos/ardilla      2        165
3      168                                  Armiño                                fauna/mamiferos/armino      2        165
4      169                                    Asno                                  fauna/mamiferos/asno      2        165
5      170                                Ballenas                              fauna/mamiferos/ballenas      2        165
6      171                   Bisonte (Bison Bison)                   fauna/mamiferos/bisonte-bison-bison      2        165
7      172      